# Sample and null permute portion of every conversation.

Requested by @RickDale

In [1]:
import os
import numpy as np
import pandas as pd
import re
import shutil

import pyarrow.parquet as pq
from tqdm.notebook import tqdm
import warnings;warnings.filterwarnings('ignore')

In [2]:
SERVER_READY_DATA = '../data/server_ready'

OUTPUT_FOLDER = '../data/server_ready_'
if not os.path.exists(OUTPUT_FOLDER):
    os.mkdir(OUTPUT_FOLDER)

In [3]:
def grab_all_files(PATH, file_type='.parquet'):
    files = [
        [
            os.path.join(root, f) for f in files
            if f.endswith(file_type) and (not f.startswith('.'))
        ]
        for root, _, files in os.walk(PATH)
    ]
    return sum(files, [])

## Iterate, permute

In [4]:
fs = grab_all_files(SERVER_READY_DATA)
len(fs)

1669

### Everything other than CANDOR

In [9]:
NON_CANDOR = [f for f in fs if (not bool(re.findall(r'CANDOR-\d+-\d+\.parquet', f)))]
len(NON_CANDOR)

13

In [10]:
families = set([f.split('/')[-1].split('-')[0].replace('.parquet', '') for f in NON_CANDOR])
families

{'CABNC',
 'CORAAL',
 'GCSAusE',
 'MICASE',
 'callfriend',
 'grace_corpus',
 'med_school'}

In [11]:
for fam in tqdm(families):

    familiars = [f for f in NON_CANDOR if (fam in f)]
    df = [
        pq.ParquetFile(f).read(columns=['corpus', 'conversation_id', 'x_turn_id', 'y_turn_id', 'x_speaker', 'y_speaker', 'x_utterance', 'y_utterance']).to_pandas()
        for f in familiars
    ]

    df = pd.concat(df, ignore_index=True)

    good_y = ~df['y_utterance'].isna()

    df['output_file_name'] = df['corpus'] + '-' + df['conversation_id'] + '.parquet'

    groups = df.groupby(by=['output_file_name'])
    for out_f,dfi in groups:

        dfi[[col for  col in list(dfi) if col != 'output_file_name']].to_parquet(
            os.path.join(OUTPUT_FOLDER, out_f[0]),
            engine='fastparquet',
            compression='snappy'
        )

  0%|          | 0/7 [00:00<?, ?it/s]

### CANDOR

In [12]:
CANDOR =[f for f in fs if bool(re.findall(r'CANDOR-\d+-\d+\.parquet', f))]
len(CANDOR)

1656

In [14]:
for convo in tqdm(CANDOR):

    df = pd.read_parquet(convo)

    df['corpus'] = 'CANDOR'

    good_y = ~df['y_utterance'].isna()

    df['output_file_name'] = df['corpus'] + '-' + df['conversation_id'] + '.parquet'

    groups = df.groupby(by=['output_file_name'])
    for out_f,dfi in groups:


        dfi[[col for  col in list(dfi) if col != 'output_file_name']].to_parquet(
            os.path.join(OUTPUT_FOLDER, out_f[0]),
            engine='fastparquet',
            compression='snappy'
        )


  0%|          | 0/1656 [00:00<?, ?it/s]

## Sample/Test

In [4]:
fs = grab_all_files(OUTPUT_FOLDER)
len(fs)

4522

In [5]:
for f in tqdm(np.random.choice(fs,size=(10,), replace=False)):
    df = pd.read_parquet(f)
    print(f.split('/')[-1])
    print(len(df))
    print(df.isna().sum())
    print(df['y_utterance'].sample(n=3).values.tolist())
    print('-----\n')

    # df.to_parquet(
    #     os.path.join('/Users/zacharyrosen/Desktop/', f.split('/')[-1]),
    #     engine='fastparquet',
    #     compression='snappy'
    # )
    # df.to_parquet(
    #     f,
    #     engine='fastparquet',
    #     compression='snappy'
    # )

  0%|          | 0/10 [00:00<?, ?it/s]

CABNC-CABNC-KB9RE016-0missing.parquet
5917
corpus             0
conversation_id    0
x_turn_id          0
y_turn_id          0
x_speaker          0
y_speaker          0
x_utterance        0
y_utterance        0
dtype: int64
['Matthew', "If I sleep during the day I do n't sleep at night", 'Sydenham High School']
-----

CANDOR-CANDOR-19-3c923944-0979-45d1-8396-a72dd9ec55d2.parquet
2030
x_turn_id          0
x_utterance        0
x_speaker          0
x_start            0
x_stop             0
x_delta            0
conversation_id    0
x_race             0
self               0
y_turn_id          0
y_utterance        0
y_speaker          0
y_start            0
y_stop             0
y_delta            0
y_race             0
corpus             0
dtype: int64
['Mhm.', "Yeah. Absolutely. Uh No. Mhm. I mean there's there's some crimes I found out that my my two times great grandfather served at the Missouri State penitentiary ah in like 1900 steve avery people yeah 1900 very important.", 'Mm. Mhm. Ho

## Sort and send

In [6]:
fs = [f for f in os.listdir(OUTPUT_FOLDER) if not f.startswith('.') and (not os.path.isdir(os.path.join(OUTPUT_FOLDER,f)))]
len(fs)

4522

In [7]:
to_rosen = np.random.choice(fs, size=(int(len(fs)/2),), replace=False).tolist()
to_itkin = [f for f in fs if f not in to_rosen]
len(to_itkin), len(to_rosen)

(2261, 2261)

In [8]:
ROSEN_PATH = os.path.join(OUTPUT_FOLDER, 'to_rosen')
if not os.path.exists(ROSEN_PATH):
    os.mkdir(ROSEN_PATH)

for f in tqdm(to_rosen):
    shutil.move(
        os.path.join(OUTPUT_FOLDER, f),
        os.path.join(ROSEN_PATH, f)
    )

  0%|          | 0/2261 [00:00<?, ?it/s]

In [9]:
ITKIN_PATH = os.path.join(OUTPUT_FOLDER, 'to_itkin')
if not os.path.exists(ITKIN_PATH):
    os.mkdir(ITKIN_PATH)

for f in tqdm(to_itkin):
    shutil.move(
        os.path.join(OUTPUT_FOLDER, f),
        os.path.join(ITKIN_PATH, f)
    )

  0%|          | 0/2261 [00:00<?, ?it/s]

##### Sort and send (OLD)

In [ ]:
CANDOR = [f for f in os.listdir(OUTPUT_FOLDER) if ('CANDOR-' in f) and (not f.startswith('.'))]
NON_CANDOR = [f for f in os.listdir(OUTPUT_FOLDER) if ('CANDOR-' not in f) and (not f.startswith('.'))]

In [ ]:
to_rosen, to_itkin = [], []
to_rosen += np.random.choice(CANDOR, size=(int(len(CANDOR)/2),), replace=False).tolist() + np.random.choice(NON_CANDOR, size=(int(len(NON_CANDOR)/2),), replace=False).tolist()
to_itkin = [f for f in CANDOR if f not in to_rosen] + [f for f in NON_CANDOR if f not in to_rosen]
len(to_itkin), len(to_rosen)

In [ ]:
ROSEN_PATH = os.path.join(OUTPUT_FOLDER, 'to_rosen')
if not os.path.exists(ROSEN_PATH):
    os.mkdir(ROSEN_PATH)

for f in tqdm(to_rosen):
    shutil.move(
        os.path.join(OUTPUT_FOLDER, f),
        os.path.join(ROSEN_PATH, f)
    )

In [ ]:
ITKIN_PATH = os.path.join(OUTPUT_FOLDER, 'to_itkin')
if not os.path.exists(ITKIN_PATH):
    os.mkdir(ITKIN_PATH)

for f in tqdm(to_itkin):
    shutil.move(
        os.path.join(OUTPUT_FOLDER, f),
        os.path.join(ITKIN_PATH, f)
    )